# 🌐 NPPE-1
### Fine-tuning Gemma 3-1B-IT with QLoRA for 13 Indic Languages

**Strategy:** 4-bit quantised Gemma 3 1B + LoRA adapters, trained on all available data,  
with few-shot prompting and majority-vote inference.

In [1]:
%%html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Gemma 3 · Multilingual Sentiment Architecture</title>
<style>
  @import url('https://fonts.googleapis.com/css2?family=Caveat:wght@400;600;700&family=Special+Elite&display=swap');

  :root {
    --bg: #fdf6e3;
    --paper: #fffef7;
    --ink: #1a1208;
    --faint: #b8a98a;
    --blue:  #2563a8;
    --green: #2a7a4b;
    --orange:#c05a10;
    --red:   #b03030;
    --purple:#6b3fa0;
    --teal:  #1a7a7a;
    --yellow:#e8c840;
    --shadow: rgba(26,18,8,0.13);
  }

  * { box-sizing: border-box; margin: 0; padding: 0; }

  body {
    background: var(--bg);
    font-family: 'Caveat', cursive;
    color: var(--ink);
    min-height: 100vh;
    padding: 24px;
    background-image:
      radial-gradient(circle at 20% 20%, rgba(37,99,168,0.04) 0%, transparent 50%),
      radial-gradient(circle at 80% 80%, rgba(107,63,160,0.04) 0%, transparent 50%),
      url("data:image/svg+xml,%3Csvg width='40' height='40' xmlns='http://www.w3.org/2000/svg'%3E%3Cpath d='M0 40 L40 0' stroke='%23b8a98a' stroke-width='0.3' opacity='0.3'/%3E%3C/svg%3E");
  }

  h1 {
    font-family: 'Special Elite', monospace;
    font-size: 26px;
    text-align: center;
    margin-bottom: 6px;
    color: var(--ink);
    letter-spacing: 1px;
  }
  .subtitle {
    text-align: center;
    font-size: 16px;
    color: var(--faint);
    margin-bottom: 28px;
    font-family: 'Caveat', cursive;
  }

  /* ── canvas ── */
  .canvas {
    position: relative;
    width: 100%;
    max-width: 1100px;
    margin: 0 auto;
    background: var(--paper);
    border: 2.5px solid var(--ink);
    border-radius: 4px;
    box-shadow: 6px 6px 0 var(--ink);
    padding: 36px 32px 44px;
    overflow: visible;
  }

  /* ── section labels (floating tape) ── */
  .tape {
    display: inline-block;
    background: var(--yellow);
    font-family: 'Special Elite', monospace;
    font-size: 11px;
    letter-spacing: 1.5px;
    text-transform: uppercase;
    padding: 3px 10px;
    border: 1.5px solid var(--ink);
    transform: rotate(-1deg);
    margin-bottom: 10px;
    box-shadow: 2px 2px 0 var(--ink);
  }

  /* ── sketch box ── */
  .box {
    position: relative;
    border: 2.5px solid var(--ink);
    border-radius: 6px;
    background: white;
    padding: 10px 14px;
    box-shadow: 3px 3px 0 var(--shadow);
    transition: transform 0.15s, box-shadow 0.15s;
    cursor: default;
  }
  .box:hover {
    transform: translate(-1px,-2px);
    box-shadow: 5px 6px 0 var(--shadow);
  }
  .box .label {
    font-size: 15px;
    font-weight: 700;
    line-height: 1.2;
  }
  .box .sub {
    font-size: 12px;
    color: #666;
    margin-top: 2px;
    line-height: 1.3;
  }
  .box .badge {
    display: inline-block;
    font-size: 10px;
    font-family: 'Special Elite', monospace;
    padding: 1px 6px;
    border: 1.5px solid currentColor;
    border-radius: 3px;
    margin-top: 4px;
    letter-spacing: 0.5px;
  }

  /* colour themes */
  .c-blue   { border-color: var(--blue);   background: #eef5ff; }
  .c-green  { border-color: var(--green);  background: #edfaf2; }
  .c-orange { border-color: var(--orange); background: #fff5eb; }
  .c-red    { border-color: var(--red);    background: #fff0f0; }
  .c-purple { border-color: var(--purple); background: #f5eeff; }
  .c-teal   { border-color: var(--teal);   background: #e8fafa; }
  .c-yellow { border-color: #b8940a;       background: #fffbe6; }
  .c-ink    { border-color: var(--ink);    background: var(--ink); color: white; }

  /* ── LAYOUT ROWS ── */
  .row {
    display: flex;
    align-items: stretch;
    gap: 16px;
    margin-bottom: 0;
  }
  .col { flex: 1; }
  .col-2 { flex: 2; }
  .col-3 { flex: 3; }

  /* ── arrows between rows ── */
  .arrow-row {
    display: flex;
    align-items: center;
    justify-content: center;
    height: 38px;
    gap: 8px;
    position: relative;
  }
  .arrow-row svg {
    display: block;
    overflow: visible;
  }
  .arrow-label {
    font-size: 12px;
    color: var(--faint);
    font-style: italic;
    white-space: nowrap;
  }

  /* ── group wrapper (dashed region) ── */
  .group {
    border: 2px dashed var(--faint);
    border-radius: 10px;
    padding: 14px 16px 16px;
    position: relative;
    margin-bottom: 0;
  }
  .group-title {
    position: absolute;
    top: -13px;
    left: 14px;
    background: var(--paper);
    padding: 0 6px;
    font-family: 'Special Elite', monospace;
    font-size: 11px;
    letter-spacing: 1px;
    text-transform: uppercase;
    color: var(--faint);
  }

  /* ── full pipeline stack ── */
  .pipeline {
    display: flex;
    flex-direction: column;
    gap: 0;
  }

  /* ── inline chip ── */
  .chip {
    display: inline-block;
    background: var(--ink);
    color: white;
    font-size: 10px;
    font-family: 'Special Elite', monospace;
    padding: 1px 7px;
    border-radius: 2px;
    letter-spacing: 0.5px;
    margin: 2px 2px 0 0;
  }
  .chip.blue   { background: var(--blue); }
  .chip.green  { background: var(--green); }
  .chip.orange { background: var(--orange); }
  .chip.purple { background: var(--purple); }
  .chip.teal   { background: var(--teal); }

  /* ── hand-drawn SVG arrows ── */
  .svg-arrows {
    position: absolute;
    top: 0; left: 0;
    width: 100%; height: 100%;
    pointer-events: none;
    overflow: visible;
  }

  /* ── legend ── */
  .legend {
    display: flex;
    flex-wrap: wrap;
    gap: 10px;
    margin-top: 24px;
    padding-top: 16px;
    border-top: 1.5px dashed var(--faint);
    font-size: 13px;
  }
  .legend-item {
    display: flex;
    align-items: center;
    gap: 6px;
  }
  .legend-dot {
    width: 12px; height: 12px;
    border: 2px solid var(--ink);
    border-radius: 2px;
  }

  /* ── annotations ── */
  .note {
    font-family: 'Caveat', cursive;
    font-size: 13px;
    color: var(--faint);
    font-style: italic;
  }

  /* ── wiggle on hover ── */
  @keyframes wiggle {
    0%,100% { transform: rotate(0deg); }
    25%      { transform: rotate(-1.5deg); }
    75%      { transform: rotate(1.5deg); }
  }
  .box:hover { animation: wiggle 0.4s ease; }

  .star {
    color: var(--orange);
    font-size: 18px;
  }

  hr.sketch {
    border: none;
    border-top: 2px dashed var(--faint);
    margin: 20px 0;
  }

  /* tooltip */
  .box[data-tip]:hover::after {
    content: attr(data-tip);
    position: absolute;
    bottom: calc(100% + 8px);
    left: 50%;
    transform: translateX(-50%);
    background: var(--ink);
    color: white;
    font-size: 12px;
    padding: 4px 10px;
    border-radius: 4px;
    white-space: nowrap;
    z-index: 99;
    pointer-events: none;
    font-family: 'Caveat', cursive;
  }
</style>
</head>
<body>

<h1>🤖 Gemma 3-1B-IT · Multilingual Sentiment Pipeline</h1>
<p class="subtitle">NPPE-1 · 13 Indic Languages · QLoRA Fine-tuning ·</p>

<div class="canvas">

  <!-- ══════════════════════════════════════════════════
       ROW 0: INPUT
  ═══════════════════════════════════════════════════ -->
  <div class="row">
    <div class="col">
      <div class="tape">① Input</div>
      <div class="row" style="gap:12px; margin-top:0;">

        <div class="col box c-blue" data-tip="13 languages, balanced Pos/Neg">
          <div class="label">📂 train.csv</div>
          <div class="sub">Labelled sentences<br>13 Indic languages</div>
          <span class="chip blue">Positive / Negative</span>
        </div>

        <div class="col box c-blue" data-tip="Unlabelled — predict these">
          <div class="label">📂 test.csv</div>
          <div class="sub">Unlabelled sentences<br>for final prediction</div>
          <span class="chip blue">No labels</span>
        </div>

        <div class="col box c-yellow" data-tip="as, bd, bn, gu, hi, kn, ml, mr, or, pa, ta, te, ur">
          <div class="label">🌐 13 Languages</div>
          <div class="sub">Assamese · Bodo · Bengali<br>Hindi · Tamil · Telugu…</div>
          <span class="chip">ISO codes</span>
        </div>

      </div>
    </div>
  </div>

  <!-- arrow -->
  <div class="arrow-row">
    <svg width="40" height="38"><path d="M20 4 Q22 19 20 34" stroke="#1a1208" stroke-width="2" fill="none" stroke-dasharray="4,2"/><polygon points="14,30 20,38 26,30" fill="#1a1208"/></svg>
    <span class="arrow-label">prompt engineering</span>
  </div>

  <!-- ══════════════════════════════════════════════════
       ROW 1: PROMPT DESIGN
  ═══════════════════════════════════════════════════ -->
  <div class="tape">② Prompt Design</div>
  <div class="group" style="margin-bottom:0;">
    <span class="group-title">Few-shot Chat Template</span>
    <div class="row" style="gap:12px;">

      <div class="col-2 box c-purple" data-tip="Gemma chat format: user/model turns">
        <div class="label">💬 Chat Format</div>
        <div class="sub"><code style="font-size:11px">&lt;start_of_turn&gt;user</code><br>Instruction + few-shot examples<br><code style="font-size:11px">&lt;start_of_turn&gt;model</code></div>
        <span class="chip purple">Gemma template</span>
      </div>

      <div class="col box c-purple" data-tip="4 examples covering Tamil, Bengali, Urdu, Telugu">
        <div class="label">🎯 Few-shot</div>
        <div class="sub">4 cross-lingual examples<br>injected into every prompt</div>
        <span class="chip purple">k=4</span>
      </div>

      <div class="col box c-purple" data-tip="Completion-only loss: gradient on answer token only">
        <div class="label">🎓 Training Label</div>
        <div class="sub">Answer appended:<br><em>"Positive"</em> or <em>"Negative"</em></div>
        <span class="chip purple">completion-only loss</span>
      </div>

    </div>
  </div>

  <!-- arrow -->
  <div class="arrow-row">
    <svg width="40" height="38"><path d="M20 4 Q22 19 20 34" stroke="#1a1208" stroke-width="2" fill="none" stroke-dasharray="4,2"/><polygon points="14,30 20,38 26,30" fill="#1a1208"/></svg>
    <span class="arrow-label">tokenise &amp; quantise</span>
  </div>

  <!-- ══════════════════════════════════════════════════
       ROW 2: MODEL LOADING
  ═══════════════════════════════════════════════════ -->
  <div class="tape">③ Model Loading</div>
  <div class="group" style="margin-bottom:0;">
    <span class="group-title">4-bit Quantised Gemma 3</span>
    <div class="row" style="gap:12px;">

      <div class="col-2 box c-ink" data-tip="google/gemma-3-1b-it — instruction-tuned">
        <div class="label" style="color:white">🔮 Gemma 3-1B-IT</div>
        <div class="sub" style="color:#ccc">Base instruction-tuned model<br>~1 billion parameters</div>
        <span class="chip" style="background:#ffffff22; border:1px solid #fff; color:white">google/gemma-3</span>
      </div>

      <div class="col box c-orange" data-tip="NF4 quant + double quant → ~0.9 GB VRAM">
        <div class="label">⚡ BitsAndBytes</div>
        <div class="sub">4-bit NF4 quant<br>FP16 compute<br>Double quantisation</div>
        <span class="chip orange">~0.9 GB VRAM</span>
      </div>

      <div class="col box c-orange" data-tip="Gradient checkpointing saves memory during training">
        <div class="label">🧠 kbit Training</div>
        <div class="sub">prepare_model_for<br>_kbit_training()<br>+ grad checkpointing</div>
        <span class="chip orange">memory opt.</span>
      </div>

    </div>
  </div>

  <!-- arrow -->
  <div class="arrow-row">
    <svg width="40" height="38"><path d="M20 4 Q22 19 20 34" stroke="#1a1208" stroke-width="2" fill="none" stroke-dasharray="4,2"/><polygon points="14,30 20,38 26,30" fill="#1a1208"/></svg>
    <span class="arrow-label">inject adapters</span>
  </div>

  <!-- ══════════════════════════════════════════════════
       ROW 3: LoRA
  ═══════════════════════════════════════════════════ -->
  <div class="tape">④ LoRA Fine-Tuning</div>
  <div class="group" style="margin-bottom:0;">
    <span class="group-title">Parameter-Efficient Fine-Tuning (PEFT)</span>
    <div class="row" style="gap:12px;">

      <div class="col box c-green" data-tip="r=32, alpha=64 — standard 2× scaling">
        <div class="label">🔩 LoRA Config</div>
        <div class="sub">rank r = <strong>32</strong><br>alpha α = <strong>64</strong><br>dropout = 0.05</div>
        <span class="chip green">CAUSAL_LM</span>
      </div>

      <div class="col-2 box c-green" data-tip="All attention + MLP projection layers targeted">
        <div class="label">🎯 Target Modules (7)</div>
        <div class="sub" style="line-height:1.8">
          <span class="chip green">q_proj</span>
          <span class="chip green">k_proj</span>
          <span class="chip green">v_proj</span>
          <span class="chip green">o_proj</span>
          <span class="chip green">gate_proj</span>
          <span class="chip green">up_proj</span>
          <span class="chip green">down_proj</span>
        </div>
      </div>

      <div class="col box c-green" data-tip="SFTTrainer from TRL library">
        <div class="label">🏋 SFTTrainer</div>
        <div class="sub">LR 2e-4 · cosine<br>8 epochs<br>batch 64 (8×8)</div>
        <span class="chip green">paged_adamw_8bit</span>
      </div>

    </div>

    <!-- sub-row: baseline eval -->
    <hr class="sketch" style="margin: 14px 0 12px;">
    <div class="row" style="gap:12px;">

      <div class="col box c-teal" data-tip="200-sample stratified split from training data">
        <div class="label">📊 Pre-FT Baseline</div>
        <div class="sub">200-sample stratified eval<br><em>before</em> adapter load</div>
        <span class="chip teal">Macro F1 baseline</span>
      </div>

      <div class="col box c-teal" data-tip="Same 200 samples re-evaluated after LoRA">
        <div class="label">📈 Post-FT Eval</div>
        <div class="sub">Same 200 samples<br><em>after</em> adapter load</div>
        <span class="chip teal">Δ improvement</span>
      </div>

      <div class="col box c-teal" data-tip="Macro F1 per language breakdown">
        <div class="label">🗺 Per-Language</div>
        <div class="sub">Accuracy breakdown<br>across all 13 languages</div>
        <span class="chip teal">lang analysis</span>
      </div>

    </div>
  </div>

  <!-- arrow -->
  <div class="arrow-row">
    <svg width="40" height="38"><path d="M20 4 Q22 19 20 34" stroke="#1a1208" stroke-width="2" fill="none" stroke-dasharray="4,2"/><polygon points="14,30 20,38 26,30" fill="#1a1208"/></svg>
    <span class="arrow-label">majority vote inference</span>
  </div>

  <!-- ══════════════════════════════════════════════════
       ROW 4: INFERENCE
  ═══════════════════════════════════════════════════ -->
  <div class="tape">⑤ Inference</div>
  <div class="group" style="margin-bottom:0;">
    <span class="group-title">Majority-Vote Decoding  ×3 Passes</span>
    <div class="row" style="gap:12px;">

      <div class="col box c-blue" data-tip="Greedy decode — deterministic">
        <div class="label">🎯 Pass 1</div>
        <div class="sub">Greedy decoding<br><code>do_sample=False</code></div>
        <span class="chip blue">deterministic</span>
      </div>

      <div class="col box c-blue" data-tip="Low temperature sampling ×2">
        <div class="label">🌡 Pass 2 &amp; 3</div>
        <div class="sub">Temperature sampling<br><code>temp = 0.3</code></div>
        <span class="chip blue">stochastic ×2</span>
      </div>

      <div class="col box c-blue" data-tip="argmax of 3 votes per sample">
        <div class="label">🗳 Majority Vote</div>
        <div class="sub">argmax across 3 runs<br>per sample</div>
        <span class="chip blue">ensemble</span>
      </div>

      <div class="col box c-blue" data-tip="Left-pad for batch generation">
        <div class="label">📐 Tokenisation</div>
        <div class="sub">Left padding<br>max_len = 256<br>batch = 16</div>
        <span class="chip blue">left-pad</span>
      </div>

    </div>
  </div>

  <!-- arrow -->
  <div class="arrow-row">
    <svg width="40" height="38"><path d="M20 4 Q22 19 20 34" stroke="#1a1208" stroke-width="2" fill="none" stroke-dasharray="4,2"/><polygon points="14,30 20,38 26,30" fill="#1a1208"/></svg>
    <span class="arrow-label">map → 0/1</span>
  </div>

  <!-- ══════════════════════════════════════════════════
       ROW 5: OUTPUT
  ═══════════════════════════════════════════════════ -->
  <div class="tape">⑥ Output</div>
  <div class="row" style="gap:12px;">

    <div class="col box c-red" data-tip="submission.csv with ID + label columns">
      <div class="label">📤 submission.csv</div>
      <div class="sub">ID · label (0 or 1)<br>Positive=1, Negative=0</div>
      <span class="chip" style="background:var(--red)">Kaggle submit</span>
    </div>

    <div class="col box c-red" data-tip="Macro-averaged F1 across both classes">
      <div class="label">🏆 Metric</div>
      <div class="sub">Macro F1 Score<br>averaged: Pos + Neg</div>
      <span class="chip" style="background:var(--red)">competition metric</span>
    </div>

    <div class="col-2 box c-yellow" data-tip="GPU: T4 · VRAM">
      <div class="label">⚙️ Hardware</div>
      <div class="sub" style="line-height:1.8">
        <span class="chip">T4 GPU </span>
        <span class="chip orange">4-bit VRAM </span>
        <span class="chip purple">PEFT adapters </span>
        <span class="chip teal">Kaggle Notebook</span>
      </div>
    </div>

  </div>

  <!-- ── LEGEND ── -->
  <div class="legend">
    <strong style="font-family:'Special Elite',monospace; font-size:12px; letter-spacing:1px; text-transform:uppercase; color:var(--faint); margin-right:4px;">Legend:</strong>
    <div class="legend-item"><div class="legend-dot" style="background:#eef5ff; border-color:var(--blue)"></div><span>Data / I/O</span></div>
    <div class="legend-item"><div class="legend-dot" style="background:#f5eeff; border-color:var(--purple)"></div><span>Prompt Engineering</span></div>
    <div class="legend-item"><div class="legend-dot" style="background:#1a1208; border-color:var(--ink)"></div><span style="color:white; background:var(--ink); padding:0 4px">Base Model</span></div>
    <div class="legend-item"><div class="legend-dot" style="background:#fff5eb; border-color:var(--orange)"></div><span>Quantisation</span></div>
    <div class="legend-item"><div class="legend-dot" style="background:#edfaf2; border-color:var(--green)"></div><span>LoRA / PEFT</span></div>
    <div class="legend-item"><div class="legend-dot" style="background:#e8fafa; border-color:var(--teal)"></div><span>Evaluation</span></div>
    <div class="legend-item"><div class="legend-dot" style="background:#eef5ff; border-color:var(--blue)"></div><span>Inference</span></div>
    <div class="legend-item"><div class="legend-dot" style="background:#fff0f0; border-color:var(--red)"></div><span>Output</span></div>
    <div class="note" style="margin-left:auto">hover any box for details ✦</div>
  </div>

</div>

<!-- ── footer ── -->
<p style="text-align:center; margin-top:14px; font-size:13px; color:var(--faint);">
  NPPE-1 · DLP 2026 Term 1 · Gemma 3-1B-IT QLoRA · <em>hover boxes for details</em>
</p>

</body>
</html>

## 0. Environment Setup

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # single T4 — required for 4-bit + PEFT

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1/config.json
/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1/tokenizer.json
/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1/tokenizer_config.json
/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1/model.safetensors
/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1/special_tokens_map.json
/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1/tokenizer.model
/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1/added_tokens.json
/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1/generation_config.json
/kaggle/input/datasets/rishabpanwar8876/gemma3-best-adapter/adapter_model.safetensors
/kaggle/input/datasets/rishabpanwar8876/gemma3-best-adapter/adapter_config.json
/kaggle/input/datasets/rishabpanwar8876/gemma3-best-adapter/README.md
/kaggle/input/datasets/rishabpanwar8876/gemma3-best-adapter/tokenizer.json
/kaggle/input/dataset

In [3]:
!pip install -q --upgrade transformers accelerate bitsandbytes peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 39.3 MB/s eta 0:00:00


In [4]:
import time
import torch
import numpy as np
import pandas as pd
from peft import PeftModel
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from trl import SFTTrainer, SFTConfig
import bitsandbytes as bnb

def track_time(start, label):
    print(f"{label} — {time.time() - start:.1f}s\n")

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


## 1. Load Data

In [5]:
t = time.time()

DATA_DIR = "/kaggle/input/competitions/nppe-dlp-2026-term-1"
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")

print(f"Train: {train_df.shape}, Test: {test_df.shape}")
print(train_df['label'].value_counts())
print(train_df['language'].value_counts())
display(train_df.head(3))

track_time(t, "Data loaded")

Train: (900, 4), Test: (100, 3)
label
Positive    456
Negative    444
Name: count, dtype: int64
language
ta    76
hi    74
or    72
pa    72
as    71
bd    71
gu    69
ml    68
mr    67
kn    66
ur    66
bn    65
te    63
Name: count, dtype: int64


,ID,sentence,label,language
0,82,ਇਹ ਫਿਲਮ ਇੱਕ ਬੇਹਤਰੀਨ ਕਹਾਣੀ ਸੁਣਾਉਣ ਦਾ ਸਭ ਤੋਂ ਵਧੀ...,Negative,pa
1,618,"ਇੱਕ ਸਮਗਰੀ ਦੇ ਰੂਪ ਵਿੱਚ, ਕੋਟਿੰਗ ਤੋਂ ਬਿਨਾਂ, ਸਿਰਫ ...",Negative,pa
2,812,"ബ്രിസിലുകൾ കട്ടിയുള്ള പ്ലാസ്റ്റിക് ആണ്, അതിനാൽ...",Negative,ml


Data loaded — 0.1s



## 2. Prompt Design

In [6]:
LANG_MAP = {
    "as": "Assamese", "bd": "Bodo",      "bn": "Bengali",
    "gu": "Gujarati",  "hi": "Hindi",     "kn": "Kannada",
    "ml": "Malayalam", "mr": "Marathi",   "or": "Odia",
    "pa": "Punjabi",   "ta": "Tamil",     "te": "Telugu",
    "ur": "Urdu"
}

RESPONSE_TEMPLATE = "<start_of_turn>model\n"

FEW_SHOT = """Examples:
- "வாடிக்கையாளர் சேவை மிகவும் நன்றாக இருந்தது" → Positive (Tamil)
- "এই পণ্যটি একদম বাজে" → Negative (Bengali)  
- "بہت اچھی سروس ہے" → Positive (Urdu)
- "ఈ సేవ చాలా చెత్తగా ఉంది" → Negative (Telugu)

"""

def make_prompt(sentence: str, language: str, label: str = None) -> str:
    lang_name = LANG_MAP.get(language, language)
    instruction = (
        f"Classify the sentiment of the following {lang_name} sentence as Positive or Negative.\n\n"
        f"{FEW_SHOT}"
        f"Now classify this:\n"
        f"Sentence: {sentence}\n"
        f"Reply with exactly one word: Positive or Negative."
    )
    if label is not None:
        return (
            f"<start_of_turn>user\n{instruction}<end_of_turn>\n"
            f"{RESPONSE_TEMPLATE}{label}<end_of_turn>"
        )
    else:
        return (
            f"<start_of_turn>user\n{instruction}<end_of_turn>\n"
            f"{RESPONSE_TEMPLATE}"
        )

## 3. Build Training Dataset

In [7]:
t = time.time()

# Use ALL training data
train_df['text'] = train_df.apply(
    lambda r: make_prompt(r['sentence'], r['language'], r['label']), axis=1
)

train_dataset = Dataset.from_pandas(train_df[['text']].reset_index(drop=True))

print(f"Training on {len(train_dataset)} samples (all data)")
print("\nExample:")
print(train_dataset[0]['text'])

track_time(t, "Dataset prepared")

Training on 900 samples (all data)

Example:
<start_of_turn>user
Classify the sentiment of the following Punjabi sentence as Positive or Negative.

Examples:
- "வாடிக்கையாளர் சேவை மிகவும் நன்றாக இருந்தது" → Positive (Tamil)
- "এই পণ্যটি একদম বাজে" → Negative (Bengali)  
- "بہت اچھی سروس ہے" → Positive (Urdu)
- "ఈ సేవ చాలా చెత్తగా ఉంది" → Negative (Telugu)

Now classify this:
Sentence: ਇਹ ਫਿਲਮ ਇੱਕ ਬੇਹਤਰੀਨ ਕਹਾਣੀ ਸੁਣਾਉਣ ਦਾ ਸਭ ਤੋਂ ਵਧੀਆ ਉਦਾਹਰਣ ਹੈ, ਪਰ ਇਹ ਸਭ ਨਕਾਰਾਤਮਕ ਅਰਥਾਂ ਵਿੱਚ ਕੀਤਾ ਗਿਆ ਹੈ। ਤੁਹਾਨੂੰ ਹਰ ਦ੍ਰਿਸ਼ ਨੂੰ ਸਮਝਣ ਲਈ ਸਮਾਂ ਕੱਢਣਾ ਹੋਵੇਗਾ!
Reply with exactly one word: Positive or Negative.<end_of_turn>
<start_of_turn>model
Negative<end_of_turn>
Dataset prepared — 0.0s



## 4. Tokenizer & Model

In [8]:
MODEL_PATH = "/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"  # right padding for training

print("Vocab size:", tokenizer.vocab_size)
print("Pad token: ", tokenizer.pad_token)

# Verify the response template tokenizes correctly
resp_ids = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
print(f"Response template token ids: {resp_ids}")
print(f"Response template decoded: {tokenizer.decode(resp_ids)}")

Vocab size: 262144
Pad token:  <eos>
Response template token ids: [105, 4368, 107]
Response template decoded: <start_of_turn>model



In [9]:
t = time.time()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="cuda:0",
    attn_implementation="eager",
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

linear_layers = [(n, m) for n, m in model.named_modules() if isinstance(m, bnb.nn.Linear4bit)]
print(f"4-bit layers: {len(linear_layers)}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

track_time(t, "Model loaded")

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

4-bit layers: 182
GPU memory used: 1.47 GB
Model loaded — 16.8s



## 5. Pre-Fine-Tuning Evaluation (Baseline)

Before applying LoRA adapters we evaluate the **vanilla Gemma 3-1B-IT model** on a  
stratified held-out sample from the training set. This gives a baseline Macro-F1  
to compare against after fine-tuning.

In [10]:
MAX_SEQ_LEN = 256

In [11]:
#Shared inference helper (used by baseline AND post-FT evaluation)

tokenizer.padding_side = "left"  # left padding for generation
model.eval()

def predict_batch(sentences, languages, batch_size=16):
    results = []
    for i in range(0, len(sentences), batch_size):
        batch_s = sentences[i : i + batch_size]
        batch_l = languages[i : i + batch_size]
        prompts = [make_prompt(s, l) for s, l in zip(batch_s, batch_l)]
        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_SEQ_LEN,
        ).to(model.device)

        votes = []
        # Run 3 passes: 1 greedy + 2 with low temperature
        for run_cfg in [
            {"do_sample": False},
            {"do_sample": True, "temperature": 0.3},
            {"do_sample": True, "temperature": 0.3},
        ]:
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, max_new_tokens=4,
                    pad_token_id=tokenizer.eos_token_id, **run_cfg
                )
            input_len = inputs["input_ids"].shape[1]
            run_preds = []
            for out in outputs:
                decoded = tokenizer.decode(out[input_len:], skip_special_tokens=True).strip().lower()
                run_preds.append("Positive" if "positive" in decoded else "Negative")
            votes.append(run_preds)

        # Majority vote across 3 runs per sample
        for j in range(len(batch_s)):
            sample_votes = [votes[r][j] for r in range(3)]
            results.append(max(set(sample_votes), key=sample_votes.count))

        if i % (batch_size * 3) == 0:
            print(f"  {min(i+batch_size, len(sentences))}/{len(sentences)}")

    return results

In [12]:
#  Build stratified baseline sample
baseline_df, _ = train_test_split(
    train_df, test_size=0.8, stratify=train_df['label'], random_state=42
)
# Cap at 200 samples to keep runtime reasonable on T4
baseline_df = baseline_df.sample(min(200, len(baseline_df)), random_state=42).reset_index(drop=True)

print(f'Baseline evaluation set: {len(baseline_df)} samples')
print(baseline_df['label'].value_counts().to_string())
print(baseline_df['language'].value_counts().to_string())

Baseline evaluation set: 180 samples
label
Positive    91
Negative    89
language
mr    18
ur    17
ta    17
gu    16
as    16
bd    15
or    13
bn    12
ml    12
kn    12
te    12
hi    11
pa     9


In [13]:
# Run baseline inference 
print('Running baseline (pre-fine-tuning) inference ...')
t = time.time()

baseline_preds = predict_batch(
    baseline_df['sentence'].tolist(),
    baseline_df['language'].tolist(),
    batch_size=16,
)

track_time(t, 'Baseline inference complete')

# Convert numeric labels to strings if needed 
true_labels = baseline_df['label'].tolist()
if isinstance(true_labels[0], (int, float)):
    inv_map = {1: 'Positive', 0: 'Negative'}
    true_labels = [inv_map[int(l)] for l in true_labels]

# Metrics
baseline_macro_f1 = f1_score(true_labels, baseline_preds, average='macro')
baseline_acc      = (np.array(true_labels) == np.array(baseline_preds)).mean()

print('  PRE-FINE-TUNING BASELINE RESULTS')
print(f'  Macro F1  : {baseline_macro_f1:.4f}')
print(f'  Accuracy  : {baseline_acc:.4f}')
print()
print(classification_report(true_labels, baseline_preds, target_names=['Negative', 'Positive']))

Running baseline (pre-fine-tuning) inference ...
  16/180
  64/180
  112/180
  160/180
Baseline inference complete — 30.8s

  PRE-FINE-TUNING BASELINE RESULTS
  Macro F1  : 0.8330
  Accuracy  : 0.8333

              precision    recall  f1-score   support

    Negative       0.80      0.89      0.84        89
    Positive       0.88      0.78      0.83        91

    accuracy                           0.83       180
   macro avg       0.84      0.83      0.83       180
weighted avg       0.84      0.83      0.83       180



In [14]:
# Per-language accuracy breakdown 
baseline_df = baseline_df.copy()
baseline_df['pred'] = baseline_preds
baseline_df['true_str'] = [inv_map[int(l)] if isinstance(l, (int, float)) else l
                            for l in baseline_df['label']]
baseline_df['correct'] = baseline_df['true_str'] == baseline_df['pred']

lang_acc = (
    baseline_df.groupby('language')['correct']
    .agg(['sum', 'count'])
    .assign(accuracy=lambda d: d['sum'] / d['count'])
    .rename(columns={'sum': 'correct', 'count': 'total'})
    .sort_values('accuracy', ascending=False)
)
lang_acc.index = lang_acc.index.map(lambda c: f"{LANG_MAP.get(c, c)} ({c})")
print('── Per-language accuracy (baseline) ──')
print(lang_acc.to_string())

# Persist results for post-FT comparison
baseline_results = {'macro_f1': baseline_macro_f1, 'accuracy': baseline_acc}
print(f'\nBaseline saved  =>  Macro F1 = {baseline_macro_f1:.4f}  |  Accuracy = {baseline_acc:.4f}')

── Per-language accuracy (baseline) ──
                correct  total  accuracy
language                                
Hindi (hi)           11     11  1.000000
Urdu (ur)            17     17  1.000000
Marathi (mr)         18     18  1.000000
Malayalam (ml)       11     12  0.916667
Telugu (te)          11     12  0.916667
Tamil (ta)           15     17  0.882353
Gujarati (gu)        14     16  0.875000
Kannada (kn)         10     12  0.833333
Punjabi (pa)          7      9  0.777778
Bengali (bn)          9     12  0.750000
Odia (or)             9     13  0.692308
Assamese (as)        11     16  0.687500
Bodo (bd)             7     15  0.466667

Baseline saved  =>  Macro F1 = 0.8330  |  Accuracy = 0.8333


## 6. LoRA

In [15]:
# r=32 gives more expressiveness than r=16 — still very memory efficient on a 1B model
# alpha=64 (2x r) maintains the standard scaling ratio
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()

In [16]:
# Load saved adapter weights
model = PeftModel.from_pretrained(
    model,
    "/kaggle/input/datasets/rishabpanwar8876/gemma3-best-adapter",
    is_trainable=False
)
model.eval()
print("Saved adapter loaded.")

Saved adapter loaded.


## Clean previous outputs before training

In [17]:
# import shutil
# for folder in ["gemma3-sft", "gemma3-lora-adapter"]:
#     path = f"/kaggle/working/{folder}"
#     if os.path.exists(path):
#         shutil.rmtree(path)
# for f in ["submission.csv", "submissions.csv"]:
#     path = f"/kaggle/working/{f}"
#     if os.path.exists(path):
#         os.remove(path)
# print("Output directory clean.")

## 7. Train config

In [18]:
# MAX_SEQ_LEN = 256

# sft_config = SFTConfig(
#     output_dir="/kaggle/working/gemma3-sft",

#     # Checkpointing
#     save_strategy="epoch",
#     save_total_limit=1,

#     # No eval dataset this time (using all data for training)
#     eval_strategy="no",

#     # Batch & Gradient
#     per_device_train_batch_size=8,
#     gradient_accumulation_steps=8,   # effective batch = 64

#     # Optimizer
#     learning_rate=2e-4,
#     lr_scheduler_type="cosine",
#     warmup_steps=50,
#     weight_decay=0.01,
#     optim="paged_adamw_8bit",

#     # Epochs: more epochs since dataset is small
#     num_train_epochs=8,

#     # Precision: off (4-bit handles it internally)
#     fp16=False,
#     bf16=False,

#     # Memory
#     gradient_checkpointing=True,
#     gradient_checkpointing_kwargs={"use_reentrant": False},
#     dataloader_pin_memory=False,

#     # Sequence
#     max_length=MAX_SEQ_LEN,
#     packing=False,
#     dataset_text_field="text",

#     # KEY: compute loss only on completion tokens, not prompt
#     # This is what Unsloth does — masks the user turn so the model
#     # only receives gradient signal from predicting Positive/Negative
#     completion_only_loss=True,

#     # Logging
#     logging_steps=10,
#     report_to="none",
# )

# trainer = SFTTrainer(
#     model=model,
#     args=sft_config,
#     train_dataset=train_dataset,
#     processing_class=tokenizer,
# )

# print("Trainer ready.")

In [19]:
# t = time.time()
# trainer.train()
# track_time(t, "Training complete")

In [20]:
# model.save_pretrained("/kaggle/working/gemma3-lora-adapter")
# tokenizer.save_pretrained("/kaggle/working/gemma3-lora-adapter")
# print("Adapter saved.")

## 8. Inference

In [21]:
t = time.time()
predictions = predict_batch(
    test_df['sentence'].tolist(),
    test_df['language'].tolist(),
)
track_time(t, "Inference complete")
print(pd.Series(predictions).value_counts())

  16/100
  64/100
  100/100
Inference complete — 22.1s

Positive    53
Negative    47
Name: count, dtype: int64


## 9. Comparision

In [22]:
print('Running post-fine-tuning evaluation on baseline sample ...')
t = time.time()

postft_preds = predict_batch(
    baseline_df['sentence'].tolist(),
    baseline_df['language'].tolist(),
    batch_size=16,
)

track_time(t, 'Post-FT evaluation complete')

postft_macro_f1 = f1_score(true_labels, postft_preds, average='macro')
postft_acc      = (np.array(true_labels) == np.array(postft_preds)).mean()

print('  POST-FINE-TUNING RESULTS')
print(f'  Macro F1  : {postft_macro_f1:.4f}')
print(f'  Accuracy  : {postft_acc:.4f}')
print()
print(classification_report(true_labels, postft_preds, target_names=['Negative', 'Positive']))

Running post-fine-tuning evaluation on baseline sample ...
  16/180
  64/180
  112/180
  160/180
Post-FT evaluation complete — 38.9s

  POST-FINE-TUNING RESULTS
  Macro F1  : 0.9332
  Accuracy  : 0.9333

              precision    recall  f1-score   support

    Negative       0.96      0.90      0.93        89
    Positive       0.91      0.97      0.94        91

    accuracy                           0.93       180
   macro avg       0.94      0.93      0.93       180
weighted avg       0.94      0.93      0.93       180



In [23]:
# Side-by-side comparison 
delta_f1  = postft_macro_f1 - baseline_results['macro_f1']
delta_acc = postft_acc      - baseline_results['accuracy']

comparison = pd.DataFrame({
    'Metric'   : ['Macro F1', 'Accuracy'],
    'Baseline' : [f"{baseline_results['macro_f1']:.4f}", f"{baseline_results['accuracy']:.4f}"],
    'After FT' : [f'{postft_macro_f1:.4f}', f'{postft_acc:.4f}'],
    'Delta'    : [f'{delta_f1:+.4f}', f'{delta_acc:+.4f}'],
})

print('  FINE-TUNING IMPACT SUMMARY')
print(comparison.to_string(index=False))

  FINE-TUNING IMPACT SUMMARY
  Metric Baseline After FT   Delta
Macro F1   0.8330   0.9332 +0.1002
Accuracy   0.8333   0.9333 +0.1000


## 10. Submission

In [24]:
label_map = {"Positive": 1, "Negative": 0}

submission = pd.DataFrame({
    'ID': test_df['ID'],
    'label': [label_map[p] for p in predictions]
})
submission.to_csv("/kaggle/working/submission.csv", index=False)